In [4]:
import pandas as pd

resolved = pd.read_parquet("../data/interim/recipes_resolved_features.parquet")

resolved[
    [
        "RecipeId",
        "RecipeServings",
        "RecipeYield",
        "RecipeServings_clean",
        "servings_parsed_from_yield",
        "ResolvedServings",
        "ResolvedServings_imputed",
        "servings_from_yield",
        "yield_unit_token",
    ]
].sample(20, random_state=18)

,RecipeId,RecipeServings,RecipeYield,RecipeServings_clean,servings_parsed_from_yield,ResolvedServings,ResolvedServings_imputed,servings_from_yield,yield_unit_token
460168,477135,NaN,1 pie,NaN,NaN,NaN,8.0,0,yield__pie
221537,230911,4.0,NaN,4.0,NaN,4.0,4.0,0,
281434,292527,NaN,40 appetizers,NaN,NaN,NaN,8.0,0,
370012,383442,NaN,NaN,NaN,NaN,NaN,4.0,0,
341223,353959,8.0,1 Pan Lasagna,8.0,NaN,8.0,8.0,0,
171258,179179,4.0,NaN,4.0,NaN,4.0,4.0,0,
430642,446605,NaN,NaN,NaN,NaN,NaN,8.0,0,
222476,231883,NaN,1 1/2 cups,NaN,NaN,NaN,6.0,0,yield__cup
389693,403709,12.0,12 bruschetta,12.0,NaN,12.0,12.0,0,
153332,160679,NaN,NaN,NaN,NaN,NaN,6.0,0,


In [5]:
derived_cook = resolved["cooktime_derived_from_total_prep"] == 1

resolved.loc[derived_cook, "CookTime_Minutes_resolved"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
)

count    82543.000000
mean         0.632531
std         16.161263
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
90%          0.000000
95%          0.000000
99%          0.000000
max       1560.000000
Name: CookTime_Minutes_resolved, dtype: float64

In [6]:
zero_derived_cook = (
    derived_cook
    & (resolved["CookTime_Minutes_resolved"] == 0)
).sum()

total_derived_cook = derived_cook.sum()

print("Derived CookTime count:", total_derived_cook)
print("Derived CookTime = 0:", zero_derived_cook)
print("Percent zero among derived:", zero_derived_cook / total_derived_cook * 100)

Derived CookTime count: 82543
Derived CookTime = 0: 81962
Percent zero among derived: 99.29612444422908


In [7]:
resolved["time_arithmetic_mismatch"].value_counts()

time_arithmetic_mismatch
0    522493
1        24
Name: count, dtype: int64

For 99.30% of recipes where CookTime was missing but could be derived, TotalTime equaled PrepTime. Under the dataset’s time equation, this implies a resolved CookTime of 0 minutes. Therefore, missing CookTime appears to represent recipes with no separately recorded cooking stage, rather than random missingness.